# Setup / build / submit the 6 NTASKS_OCN scaling cases

Runs `./case.setup`, `./preview_run`, `./case.build`, `./case.submit` for each `scaling_n{N}` case
(N = 1, 2, 4, 5, 10, 20) as fresh subprocesses, streaming output live — same rationale as
`create_remaining_cases.ipynb`: avoids blocking silently inside an MCP tool call for the slow steps
(`case.build` especially, which compiles the model and can take 10-30 min per case).

Run cells one at a time, in order (setup+preview for all 6 first, then build, then submit).
If a cell seems stuck, interrupt just that cell (kernel stays alive) and re-run it.
`case.submit` itself returns quickly (it just calls `qsub`) — use `qstat -u $USER` or the
`get_job_status`/`tail_log` MCP tools afterward to watch the actual run in the `develop` queue.

Kernel: `CrocoDash` conda env.

In [ ]:
import subprocess
from pathlib import Path

CROC_CASES = Path("/glade/u/home/manishrv/croc_cases")
NTASKS_VALUES = [1, 2, 4, 5, 10, 20]

def run_script(ntasks_ocn: int, script: str, args=None):
    caseroot = CROC_CASES / f"scaling_n{ntasks_ocn}"
    cmd = [f"./{script}"] + (args or [])
    print(f"$ (cd {caseroot} && {' '.join(cmd)})", flush=True)
    proc = subprocess.Popen(
        cmd,
        cwd=caseroot,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f"{script} for scaling_n{ntasks_ocn} exited {ret}")

def setup_and_preview(ntasks_ocn: int):
    run_script(ntasks_ocn, "case.setup")
    run_script(ntasks_ocn, "preview_run")

def build(ntasks_ocn: int):
    run_script(ntasks_ocn, "case.build")

def submit(ntasks_ocn: int):
    run_script(ntasks_ocn, "case.submit")

## Step 1 — `case.setup` + `preview_run` for all 6 cases

`preview_run` output should show the PE layout matching the requested `NTASKS_OCN` and
`JOB_QUEUE=develop` before we spend build time — sanity-check each one before moving on.

In [ ]:
setup_and_preview(1)

In [ ]:
setup_and_preview(2)

In [ ]:
setup_and_preview(4)

In [ ]:
setup_and_preview(5)

In [ ]:
setup_and_preview(10)

In [ ]:
setup_and_preview(20)

## Step 2 — `case.build` for all 6 cases

Slowest step (compiles MOM6/CESM from scratch per case). Run one at a time and watch the tail
of the output for `MODEL BUILD HAS FINISHED SUCCESSFULLY`.

In [ ]:
build(1)

In [ ]:
build(2)

In [ ]:
build(4)

In [ ]:
build(5)

In [ ]:
build(10)

In [ ]:
build(20)

## Step 3 — `case.submit` for all 6 cases

Each call just `qsub`s the job to the `develop` queue and returns — it does not wait for the
run to finish. After submitting all 6, monitor with `qstat -u $USER` (shell) or the
`get_job_status` / `tail_log` cesm-runner MCP tools. Once a run finishes, its
`timing/cesm_timing.$CASE.*` file appears — that's what `read_cesm_timing` (ocean-debugger MCP)
reads to get the scaling numbers.

In [ ]:
submit(1)

In [ ]:
submit(2)

In [ ]:
submit(4)

In [ ]:
submit(5)

In [ ]:
submit(10)

In [ ]:
submit(20)